# 02 — Gold Feature Engineering

**Input :** `hive_metastore.gold.gold_dataset`  
**Output:** `hive_metastore.gold.gold_features`

### Label design — look-ahead binary

Following the framing of Ahmadi et al. (2024, *Smart Grids and Sustainable Energy*)  
who predicted distribution-transformer overload within 1–3 hour horizons, we define  
two binary labels based on a **forward-looking** window (no data leakage):

| Column | Definition |
|---|---|
| `label_4h`  | 1 if `max(load_ratio_c, load_ratio_v) > 1` in any of the next **16 steps** (4 h)  |
| `label_24h` | 1 if `max(load_ratio_c, load_ratio_v) > 1` in any of the next **96 steps** (24 h) |

Both horizons are evaluated to quantify the precision–recall trade-off that comes  
with extending lead time (Pinci et al., 2024; ScienceDirect).

### Engineered features

| Group | Columns |
|---|---|
| Load ratio | `load_ratio_c`, `load_ratio_v` |
| Rolling stats | `{current,voltage}_{mean,std,max}_{1h,1d,7d}` (look-back only) |
| Lag features | `{current,voltage}_lag_{15m,1h,1d}` |
| Weather derived | `temp_mean_1d`, `temp_mean_7d`, `precip_sum_1d` |
| Temporal | `hour`, `day_of_week`, `month`, `is_weekend` |

> **Leakage note:** all rolling and lag features use `rowsBetween(-n, -1)` —  
> the current row is excluded. Labels use `rowsBetween(1, H)` — strictly future rows.

In [0]:
%run /Workspace/Users/daniel.branco@cgi.com/Tese/00_Utils

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

ID  = "ID_prefix"
TS  = "DATE"

SRC = "hive_metastore.gold.gold_dataset"
DST = "hive_metastore.gold.gold_features"

# Horizon definitions (15-min cadence)
H_4H  = 16   # 4 hours  = 16 steps
H_24H = 96   # 24 hours = 96 steps

SIGNAL_COLS = ["current", "voltage"]

ROLLING_WINDOWS = {
    "1h":  4,    # 1 hour
    "1d":  96,   # 1 day
    "7d":  672,  # 7 days
}

LAG_STEPS = {
    "15m": 1,
    "1h":  4,
    "1d":  96,
}

## 1 · Load gold_dataset

In [0]:
df = spark.read.table(SRC)
print(f"Loaded {df.count():,} rows  |  {len(df.columns)} columns")
df.printSchema()

## 2 · Load ratio (normalised loading)

`load_ratio_c = current / H_LIM_C`  
`load_ratio_v = voltage / H_LIM_V`

Values > 1.0 indicate the transformer is above its alarm threshold.  
These are the most informative single features and the basis for both labels.

In [0]:
df = (
    df
    .withColumn(
        "load_ratio_c",
        F.when(F.col("H_LIM_C") > 0, F.col("current") / F.col("H_LIM_C")).otherwise(F.lit(None))
    )
    .withColumn(
        "load_ratio_v",
        F.when(F.col("H_LIM_V") > 0, F.col("voltage") / F.col("H_LIM_V")).otherwise(F.lit(None))
    )
)

display(
    df.select(ID, TS, "current", "H_LIM_C", "load_ratio_c",
                      "voltage", "H_LIM_V", "load_ratio_v").limit(10)
)

## 3 · Look-ahead binary labels

For each row at time **t**, the label = 1 if the transformer will breach its  
alarm threshold (load_ratio > 1 on current **or** voltage) at **any** point  
in the look-ahead window `[t+1, t+H]`.

The forward window uses `rowsBetween(1, H)` — strictly after the current row —  
so no information from the future leaks into the features.

```
label_4h  : H = 16 steps (4 h)   — short horizon, high precision
label_24h : H = 96 steps (24 h)  — day-ahead,    lower precision
```

In [0]:
# Instantaneous overload flag (1 when either signal is above its limit)
df = df.withColumn(
    "_is_overloaded",
    (
        (F.col("load_ratio_c") > 1) | (F.col("load_ratio_v") > 1)
    ).cast("int")
)

In [0]:
def make_lookahead_label(df, horizon_steps, out_col):
    """
    Binary label: 1 if the transformer will be overloaded
    in any of the next `horizon_steps` rows (strictly future — no leakage).
    """
    w_future = (
        Window
        .partitionBy(ID)
        .orderBy(F.col(TS).cast("long"))
        .rowsBetween(1, horizon_steps)   # [t+1 … t+H], current row excluded
    )
    return df.withColumn(
        out_col,
        F.max("_is_overloaded").over(w_future).cast("int")
    )


df = make_lookahead_label(df, H_4H,  "label_4h")
df = make_lookahead_label(df, H_24H, "label_24h")

# Drop the helper column
df = df.drop("_is_overloaded")

print("Label distribution — 4h horizon:")
df.groupBy("label_4h").count().orderBy("label_4h").show()

print("Label distribution — 24h horizon:")
df.groupBy("label_24h").count().orderBy("label_24h").show()

In [0]:
# Rows at the tail of each transformer's history have no future window → label is null.
# Drop these rows — they cannot be used for supervised training.
before = df.count()
df = df.filter(F.col("label_4h").isNotNull() & F.col("label_24h").isNotNull())
after  = df.count()
print(f"Dropped {before - after:,} tail rows with null labels  |  {after:,} rows remaining")

## 4 · Rolling statistics (look-back, no leakage)

| Window name | Steps | Duration |
|---|---|---|
| 1h | 4 | 1 hour |
| 1d | 96 | 1 day |
| 7d | 672 | 7 days |

Each window uses `rowsBetween(-n, -1)` — strictly before the current row.

In [0]:
for w_name, n_rows in ROLLING_WINDOWS.items():
    w_back = (
        Window
        .partitionBy(ID)
        .orderBy(F.col(TS).cast("long"))
        .rowsBetween(-n_rows, -1)   # look-back only
    )
    for col in SIGNAL_COLS:
        df = (
            df
            .withColumn(f"{col}_mean_{w_name}", F.mean(col).over(w_back))
            .withColumn(f"{col}_std_{w_name}",  F.stddev(col).over(w_back))
            .withColumn(f"{col}_max_{w_name}",  F.max(col).over(w_back))
        )

print(f"Rolling features added ✅  — {len(df.columns)} total columns")
roll_cols = [c for c in df.columns if any(w in c for w in ROLLING_WINDOWS)]
display(df.select([ID, TS] + roll_cols[:8]).limit(5))

## 5 · Lag features

| Lag name | Steps | Lead time |
|---|---|---|
| 15m | 1 | Previous reading |
| 1h  | 4 | 1 hour ago |
| 1d  | 96 | 24 hours ago |

In [0]:
w_order = Window.partitionBy(ID).orderBy(TS)

for lag_name, n in LAG_STEPS.items():
    for col in SIGNAL_COLS:
        df = df.withColumn(f"{col}_lag_{lag_name}", F.lag(col, n).over(w_order))

print(f"Lag features added ✅  — {len(df.columns)} total columns")
lag_cols = [c for c in df.columns if "_lag_" in c]
display(df.select([ID, TS] + lag_cols).limit(5))

## 6 · Weather derived features

Raw hourly weather values are kept as-is. Three derived features are added on top:

| Feature | Window | Rationale |
|---|---|---|
| `temp_mean_1d` | 24 h look-back | Captures same-day thermal load on the transformer |
| `temp_mean_7d` | 7 d look-back | Captures hot-spell effect — sustained warmth prevents overnight cooling (Ward, 2013; cited in thesis §2.2) |
| `precip_sum_1d` | 24 h look-back | Cumulative precipitation proxies storm episodes better than a single hourly reading |

Wind and humidity are kept as instantaneous readings only — their per-hour values  
already carry the relevant signal (storm gusts, insulation stress) without needing aggregation.

In [0]:
TEMP_COL   = "temperatura_media_do_ar_horaria_c"
PRECIP_COL = "precipitacao_horaria_mm"

w_back_1d = (
    Window
    .partitionBy(ID)
    .orderBy(F.col(TS).cast("long"))
    .rowsBetween(-96, -1)    # 24 h look-back, current row excluded
)

w_back_7d = (
    Window
    .partitionBy(ID)
    .orderBy(F.col(TS).cast("long"))
    .rowsBetween(-672, -1)   # 7 d look-back, current row excluded
)

df = (
    df
    .withColumn("temp_mean_1d",   F.mean(TEMP_COL).over(w_back_1d))
    .withColumn("temp_mean_7d",   F.mean(TEMP_COL).over(w_back_7d))
    .withColumn("precip_sum_1d",  F.sum(PRECIP_COL).over(w_back_1d))
)

print("Weather derived features added ✅")
display(
    df.select(
        ID, TS,
        TEMP_COL, "temp_mean_1d", "temp_mean_7d",
        PRECIP_COL, "precip_sum_1d"
    ).limit(10)
)

## 8 · Temporal features

In [0]:
df = (
    df
    .withColumn("hour",        F.hour(TS))
    .withColumn("day_of_week", F.dayofweek(TS))   # 1 = Sunday … 7 = Saturday
    .withColumn("month",       F.month(TS))
    .withColumn("is_weekend",  F.dayofweek(TS).isin([1, 7]).cast("int"))
)

display(df.select(ID, TS, "hour", "day_of_week", "month", "is_weekend").limit(5))

## 9 · Quality checks

In [0]:
print(f"Rows : {df.count():,}")
print(f"Cols : {len(df.columns)}")
print()
df.printSchema()

In [0]:
import builtins

# Label distributions + class imbalance ratio
total = df.count()

for label_col in ["label_4h", "label_24h"]:
    dist = (
        df.groupBy(label_col).count()
          .withColumn("pct", F.round(F.col("count") / total * 100, 2))
          .orderBy(label_col)
    )
    print(f"\n=== {label_col} ===")
    dist.show()

    n_pos = df.filter(F.col(label_col) == 1).count()
    n_neg = total - n_pos
    ratio = n_neg / builtins.max(n_pos, 1)
    print(f"  Positives : {n_pos:,}")
    print(f"  Negatives : {n_neg:,}")
    print(f"  Imbalance ratio (neg/pos): {ratio:.1f}x")
    print(f"  → Suggested scale_pos_weight for XGBoost: {ratio:.1f}")
    print(f"  → Suggested pos_weight for BCEWithLogitsLoss (LSTM): {ratio:.1f}")

In [0]:
# Null rates — lag/rolling cols will have nulls at the start of each transformer's history
# (expected behaviour, not a data quality problem)
null_counts = df.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]
).toPandas().T
null_counts.columns = ["null_count"]
null_counts["null_pct"] = (null_counts["null_count"] / total * 100).round(2)

cols_with_nulls = null_counts[null_counts["null_count"] > 0].sort_values("null_pct", ascending=False)
print(f"Columns with nulls: {len(cols_with_nulls)}")
display(cols_with_nulls)

In [0]:
# Leakage sanity check:
# When the transformer is NOT currently overloaded, label_4h should not be trivially 1.
# If label_4h == 1 for a large fraction of rows where current load_ratio <= 1,
# that is expected (it means overload is coming) — not a leakage sign.
# True leakage would show label_4h == 1 = 100% when load_ratio > 1 AND == 0 otherwise.

check = df.select(
    ((F.col("load_ratio_c") > 1) | (F.col("load_ratio_v") > 1)).alias("currently_overloaded"),
    "label_4h",
    "label_24h"
)

print("=== Label rates by current overload status ===")
check.groupBy("currently_overloaded").agg(
    F.round(F.mean("label_4h"),  3).alias("label_4h_rate"),
    F.round(F.mean("label_24h"), 3).alias("label_24h_rate"),
    F.count("*").alias("n_rows")
).orderBy("currently_overloaded").show()

# Expected: label rates for currently_overloaded=False should be > 0
# (the model needs to learn from pre-overload patterns, not just current state)

In [0]:
dbutils.data.summarize(df)

## 10 · Save to `gold_features`

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS hive_metastore.gold")

(
    df.write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable(DST)
)

n_rows = spark.read.table(DST).count()
n_cols = len(spark.read.table(DST).columns)
print(f"\n✅  Saved {DST}")
print(f"   {n_rows:,} rows  |  {n_cols} columns")
print(f"\n   Labels   : label_4h (4-hour horizon), label_24h (24-hour horizon)")
print(f"   Features : load ratios, rolling stats (1h/1d/7d), lags (15m/1h/1d), weather derived (temp_mean_1d/7d, precip_sum_1d), temporal")